# Classify Land Use Intensity and Calc Proportions per 1km grid cells


In [1]:
import rasterio
import numpy as np
from rasterio.windows import Window
import gc
from pathlib import Path

# Base directory
base_path = Path("/home/georg/data/LEON_P5_BII")

## Load Data

### multiple in one variable

In [2]:
# Base directory
BASE_DIR = base_path / "EO_data_prep"

# Define all file paths in a structured dictionary
FILES = {
    'dnk': {
        2023: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_dnk_2023.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2023.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2023.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2025_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2023_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif" 
        },
        2022: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_dnk_2022.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2022.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2022.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2022_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2021: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_dnk_2021.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2021.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2021.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2021_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2020: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_dnk_2020.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2020.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2020.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2020_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2019: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_dnk_2019.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2019.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2019_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2018: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_dnk_2018.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2018.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2015_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2018_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        }
    },
    'nld': {
        2023: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_nld_2023.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2023.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2023.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2025_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2023_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2022: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_nld_2022.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2022.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2022.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2022_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2021: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_nld_2021.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2021.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2021.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2021_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2020: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_nld_2020.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2020.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2020.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2020_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2019: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_nld_2019.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2019.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2019_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2018: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_natgrass_nld_2018.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2018.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2015_3035.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2018_ghsl.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        }
    }
}

# access files like:
# FILES['dnk'][2023]['lc']
# FILES['nld'][2021]['eca']

## Batch process Land Use Intensity

In [5]:
import rasterio
import numpy as np
from pathlib import Path

##____Configuration____##

country = "dnk"
year = 2023

# Assumes FILES dict exists
file_paths = FILES[country][year]

print("="*70)
print("CHECKING LAYER ALIGNMENT AND RESOLUTION")
print("="*70)
print()

layers_to_check = {
    'lc': file_paths['lc'],
    'nt': file_paths['nt'],
    'ghsl': file_paths['ghsl'],
    'eca': file_paths['eca'],
    'rsd': file_paths['rsd'],
}

layer_info = {}

for name, path in layers_to_check.items():
    with rasterio.open(path) as src:
        layer_info[name] = {
            'shape': src.shape,
            'bounds': src.bounds,
            'crs': src.crs,
            'resolution': (src.transform[0], src.transform[4]),
            'dtype': src.dtypes[0]
        }
        print(f"{name.upper()}")
        print(f"  Shape: {src.shape}")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
        print(f"  Pixel size: {abs(src.transform[0]):.2f}m × {abs(src.transform[4]):.2f}m")
        print(f"  Data type: {src.dtypes[0]}")
        print()

print("="*70)
print("SCALE FACTOR ANALYSIS")
print("="*70)
print()

lc_h, lc_w = layer_info['lc']['shape']

for name in ['nt', 'ghsl', 'eca', 'rsd']:
    aux_h, aux_w = layer_info[name]['shape']
    scale_h = lc_h / aux_h
    scale_w = lc_w / aux_w
    
    pixel_ratio = abs(layer_info[name]['resolution'][0]) / abs(layer_info['lc']['resolution'][0])
    
    print(f"{name.upper()} relative to LC:")
    print(f"  Scale factor: {scale_h:.2f}× vertical, {scale_w:.2f}× horizontal")
    print(f"  Pixel size ratio: {pixel_ratio:.2f}×")
    print(f"  LC shape: {lc_h}×{lc_w}, {name.upper()} shape: {aux_h}×{aux_w}")
    
    if scale_h != int(scale_h) or scale_w != int(scale_w):
        print(f"  ⚠️  WARNING: Non-integer scale factor! This will cause index misalignment!")
    
    if layer_info[name]['crs'] != layer_info['lc']['crs']:
        print(f"  ⚠️  WARNING: Different CRS! LC={layer_info['lc']['crs']}, {name.upper()}={layer_info[name]['crs']}")
    
    if layer_info[name]['bounds'] != layer_info['lc']['bounds']:
        print(f"  ⚠️  WARNING: Different bounds!")
        print(f"    LC bounds: {layer_info['lc']['bounds']}")
        print(f"    {name.upper()} bounds: {layer_info[name]['bounds']}")
    
    print()

print("="*70)
print("DIAGNOSTIC: Read sample values from a test pixel")
print("="*70)
print()

# Pick a pixel in the middle of the LC layer
test_row = lc_h // 2
test_col = lc_w // 2

with rasterio.open(file_paths['lc']) as src:
    lc_val = src.read(1)[test_row, test_col]
    print(f"Test pixel location: row={test_row}, col={test_col}")
    print(f"LC value: {lc_val}")

with rasterio.open(file_paths['nt']) as src:
    scale_h = lc_h / src.shape[0]
    scale_w = lc_w / src.shape[1]
    
    nt_row = int(test_row / scale_h)
    nt_col = int(test_col / scale_w)
    nt_val = src.read(1)[nt_row, nt_col]
    
    print(f"\nNT pixel location (scaled): row={nt_row}, col={nt_col}")
    print(f"NT value: {nt_val}")
    print(f"  (scale factors: {scale_h:.2f}×, {scale_w:.2f}×)")

with rasterio.open(file_paths['ghsl']) as src:
    scale_h = lc_h / src.shape[0]
    scale_w = lc_w / src.shape[1]
    
    ghsl_row = int(test_row / scale_h)
    ghsl_col = int(test_col / scale_w)
    ghsl_val = src.read(1)[ghsl_row, ghsl_col]
    
    print(f"\nGHSL pixel location (scaled): row={ghsl_row}, col={ghsl_col}")
    print(f"GHSL value: {ghsl_val}")
    print(f"  (scale factors: {scale_h:.2f}×, {scale_w:.2f}×)")

print()
print("="*70)
print("RECOMMENDATION")
print("="*70)
print()
print("If you see:")
print("  ✓ Integer scale factors → indexing should work")
print("  ✓ Same CRS for all layers → good")
print("  ✓ Same bounds for all layers → good")
print()
print("If you see:")
print("  ✗ Non-integer scale factors → PROBLEM!")
print("    Solution: Resample all auxiliary layers to LC resolution")
print("  ✗ Different CRS → PROBLEM!")
print("    Solution: Reproject all to same CRS")
print("  ✗ Different bounds → PROBLEM!")
print("    Solution: Clip/align all to same extent")

CHECKING LAYER ALIGNMENT AND RESOLUTION

LC
  Shape: (35361, 45093)
  CRS: EPSG:3035
  Bounds: BoundingBox(left=4199570.0, bottom=3496520.0, right=4650500.0, top=3850130.0)
  Pixel size: 10.00m × 10.00m
  Data type: uint8

NT
  Shape: (3873, 4939)
  CRS: EPSG:3035
  Bounds: BoundingBox(left=4199483.700156914, bottom=3496465.32244572, right=4650583.462733572, top=3850202.7957114414)
  Pixel size: 91.33m × 91.33m
  Data type: float32

GHSL
  Shape: (4046, 5329)
  CRS: EPSG:3035
  Bounds: BoundingBox(left=4182586.867584191, bottom=3494729.9720733864, right=4669306.979908753, top=3864268.2671503574)
  Pixel size: 91.33m × 91.33m
  Data type: float64

ECA
  Shape: (353, 450)
  CRS: EPSG:3035
  Bounds: BoundingBox(left=4199570.0, bottom=3497130.0, right=4649570.0, top=3850130.0)
  Pixel size: 1000.00m × 1000.00m
  Data type: float32

RSD
  Shape: (1179, 1504)
  CRS: EPSG:3035
  Bounds: BoundingBox(left=4199400.0, bottom=3496500.0, right=4650600.0, top=3850200.0)
  Pixel size: 300.00m × 300.0

In [5]:
import numpy as np
import rasterio
from rasterio.windows import Window, bounds
import gc
from pathlib import Path

# Assumes base_path and FILES are already defined

##___Configuration___##
version = 'v3'  
countries = ['dnk', 'nld'] # , 'nld'
years = [2018, 2019, 2020, 2021, 2022, 2023] # 2018, 2019, 2020, 2021, 2022, 

# Paths
OUTPUT_DIR = base_path / "BII_LU_layer" / "Land_use_map" / version
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thresholds
THRESHOLDS = {
    'sv': {'low': 10, 'high': 40},
    'plantation': {'low': 0.2, 'high': 0.4},
    'crop': {'bare_low': 60, 'bare_high': 90, 'swf_high': 1000},
    'pasture': {'low': 0.15, 'high': 0.60},
    'urban': {'nt_thresh1': 2, 'nt_thresh2': 10, 'nt_thresh3': 20, 
              'ghsl_thresh1': 20, 'ghsl_thresh2': 100}
}

NO_DATA_CLASSES = [8, 10, 11, 253, 254, 255]
NO_DATA_VALUE = 255
CHUNK_SIZE = 2000

# ============================================
# Helper Functions
# ============================================

def read_window_from_layer(src, lc_window, lc_transform):
    """
    Read data from auxiliary layer for a given LC window.
    Handles different resolutions by mapping LC bounds to aux layer coordinates.
    Resizes output to match LC window shape.
    """
    from scipy.ndimage import zoom
    
    # Get bounds of LC window in coordinate space
    lc_bounds = bounds(lc_window, lc_transform)
    
    # Map those bounds to auxiliary layer's window
    aux_window = rasterio.windows.from_bounds(*lc_bounds, src.transform)
    
    try:
        data = src.read(1, window=aux_window)
    except Exception as e:
        # If window is out of bounds, return zeros
        data = np.zeros((lc_window.height, lc_window.width))
        return data
    
    # Resize to match LC window shape
    if data.shape != (lc_window.height, lc_window.width):
        scale_y = lc_window.height / data.shape[0]
        scale_x = lc_window.width / data.shape[1]
        data = zoom(data, (scale_y, scale_x), order=0)  # order=0 = nearest neighbor
    
    return data

def process_chunk(lc, aux_data, thresholds):
    """Process a single chunk of land cover data"""
    eca, bare, ghsl, swf, rsd, nt, sv = aux_data
    chunk_intensity = lc.copy().astype(np.uint8)
    
    # Set No Data classes first
    no_data_mask = np.isin(lc, NO_DATA_CLASSES)
    chunk_intensity[no_data_mask] = NO_DATA_VALUE
    
    # SV (Semi-Natural Vegetation)
    sv_mask = np.isin(lc, [21, 22, 23, 24])
    if np.any(sv_mask):
        eca_vals = eca[sv_mask]
        t = thresholds['sv']
        intensity = np.where(eca_vals > t['high'], 0, np.where(eca_vals >= t['low'], 1, 2))
        chunk_intensity[sv_mask] = lc[sv_mask] * 10 + intensity
    
    # Plantation
    pl_mask = (lc == 3)
    if np.any(pl_mask):
        sv_vals = sv[pl_mask]
        t = thresholds['plantation']
        intensity = np.where(sv_vals < t['low'], 2, np.where(sv_vals <= t['high'], 1, 0))
        chunk_intensity[pl_mask] = 30 + intensity
    
    # Crop
    crop_mask = np.isin(lc, [6, 7])
    if np.any(crop_mask):
        bare_vals = bare[crop_mask]
        swf_vals = swf[crop_mask]
        t = thresholds['crop']
        minimal = (bare_vals < t['bare_low']) & (swf_vals > t['swf_high'])
        light = ((bare_vals < t['bare_low']) & (swf_vals <= t['swf_high'])) | \
                ((bare_vals < t['bare_high']) & (swf_vals > t['swf_high']))
        intensity = np.where(minimal, 0, np.where(light, 1, 2))
        chunk_intensity[crop_mask] = 70 + intensity
    
    # Pasture
    pasture_mask = (lc == 41)
    if np.any(pasture_mask):
        rsd_vals = rsd[pasture_mask]
        t = thresholds['pasture']
        intensity = np.where(rsd_vals < t['low'], 0, np.where(rsd_vals <= t['high'], 1, 2))
        chunk_intensity[pasture_mask] = 60 + intensity
    
    # Urban
    urban_mask = np.isin(lc, [1, 9, 31])
    if np.any(urban_mask):
        nt_vals = nt[urban_mask]
        ghsl_vals = ghsl[urban_mask]
        t = thresholds['urban']
        
        nt_class = np.where(nt_vals < t['nt_thresh1'], 4,
                           np.where(nt_vals < t['nt_thresh2'], 3,
                                   np.where(nt_vals < t['nt_thresh3'], 2, 1)))
        
        intensity = np.where(
            nt_class == 4, 0,
            np.where(
                (nt_class == 3) & (ghsl_vals < t['ghsl_thresh1']), 0,
                np.where(
                    (nt_class == 3) & (ghsl_vals >= t['ghsl_thresh1']), 1,
                    np.where(
                        (nt_class == 2) & (ghsl_vals < t['ghsl_thresh2']), 1,
                        np.where(
                            (nt_class == 2) & (ghsl_vals >= t['ghsl_thresh2']), 2,
                            np.where(nt_class == 1, 2, 0)
                        )
                    )
                )
            )
        )
        
        chunk_intensity[urban_mask] = 10 + intensity
    
    return chunk_intensity

# ============================================
# Main Processing Loop
# ============================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for country in countries:
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing {country.upper()} {year}")
        print('='*60)
        
        file_paths = FILES[country][year]
        
        print("Starting classification...")
        with rasterio.open(file_paths['lc']) as lc_src:
            profile = lc_src.profile.copy()
            profile.update(compress='lzw', nodata=NO_DATA_VALUE)
            height, width = lc_src.shape
            lc_transform = lc_src.transform
            
            output_file = OUTPUT_DIR / f"lu_intensity_{country}_{year}_{version}.tif"
            
            # Open all auxiliary layers (once)
            aux_sources = {
                'eca': rasterio.open(file_paths['eca']),
                'bare': rasterio.open(file_paths['bare']),
                'ghsl': rasterio.open(file_paths['ghsl']),
                'swf': rasterio.open(file_paths['swf']),
                'rsd': rasterio.open(file_paths['rsd']),
                'nt': rasterio.open(file_paths['nt']),
                'sv': rasterio.open(file_paths['sv']),
            }
            
            try:
                with rasterio.open(output_file, 'w', **profile) as dst:
                    for row_start in range(0, height, CHUNK_SIZE):
                        row_end = min(row_start + CHUNK_SIZE, height)
                        
                        lc_window = Window(0, row_start, width, row_end - row_start)
                        lc = lc_src.read(1, window=lc_window)
                        
                        # Read auxiliary data for this window
                        aux_data = tuple(
                            read_window_from_layer(src, lc_window, lc_transform)
                            for src in [aux_sources['eca'], aux_sources['bare'], 
                                       aux_sources['ghsl'], aux_sources['swf'],
                                       aux_sources['rsd'], aux_sources['nt'], 
                                       aux_sources['sv']]
                        )
                        
                        chunk_intensity = process_chunk(lc, aux_data, THRESHOLDS)
                        dst.write(chunk_intensity, 1, window=lc_window)
                        
                        progress = min((row_end / height) * 100, 100)
                        print(f"  {progress:.1f}%", end='\r')
                        
                        del lc, aux_data, chunk_intensity
                        gc.collect()
                
                print(f"\n✅ Saved: {output_file}")
            
            finally:
                # Close all sources
                for src in aux_sources.values():
                    src.close()
            
            gc.collect()

print("\n🎯 All processing complete!")


Processing DNK 2018
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v3/lu_intensity_dnk_2018_v3.tif

Processing DNK 2019
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v3/lu_intensity_dnk_2019_v3.tif

Processing DNK 2020
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v3/lu_intensity_dnk_2020_v3.tif

Processing DNK 2021
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v3/lu_intensity_dnk_2021_v3.tif

Processing DNK 2022
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v3/lu_intensity_dnk_2022_v3.tif

Processing DNK 2023
Starting classification...
  100.0%
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v3/lu_intensity_dnk_2023_v3.tif

Processing NLD 2018
Starting classification...
  100.0%
✅ Saved: /home/georg/data

In [ ]:
# import numpy as np
# import rasterio
# from rasterio.windows import Window
# import gc
# from pathlib import Path

# # Assumes base_path and FILES are already defined

# ##___Configuration___##
# version = 'v2'  
# countries = ['dnk', 'nld']
# years = [2018, 2019, 2020, 2021, 2022, 2023]

# # Paths
# OUTPUT_DIR = base_path / "BII_LU_layer" / "Land_use_map" / version
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # Thresholds
# THRESHOLDS = {
#     'sv': {'low': 10, 'high': 40}, # fragmentation thresholds
#     'plantation': {'low': 0.2, 'high': 0.4},  # structural diversity thresholds
#     'crop': {'bare_low': 60, 'bare_high': 90, 'swf_high': 1000}, # bare soil days and small woody features thresholds
#     'pasture': {'low': 0.15, 'high': 0.60}, # RSD thresholds 
#     'urban': {'nt_thresh1': 2, 'nt_thresh2': 10, 'nt_thresh3': 20, 
#               'ghsl_thresh1': 10, 'ghsl_thresh2': 100} # nightlight and population thresholds
# }

# # Classes to set as No Data
# NO_DATA_CLASSES = [8, 10, 11, 253, 254, 255]
# NO_DATA_VALUE = 255

# CHUNK_SIZE = 2000

# # ============================================
# # Helper Functions
# # ============================================
# def load_auxiliary_layers(files):
#     """Load all auxiliary layers for a given year/country"""
#     with rasterio.open(files['eca']) as src:
#         eca = src.read(1)
#     with rasterio.open(files['bare']) as src:
#         bare = src.read(1)
#     with rasterio.open(files['ghsl']) as src:
#         ghsl = src.read(1)
#     with rasterio.open(files['swf']) as src:
#         swf = src.read(1)
#     with rasterio.open(files['rsd']) as src:
#         rsd = np.nan_to_num(src.read(1), nan=0.0)
#     with rasterio.open(files['nt']) as src:
#         nt = src.read(1)
#     with rasterio.open(files['sv']) as src:
#         sv = src.read(1)
#     return eca, bare, ghsl, swf, rsd, nt, sv

# def calculate_scale_factors(height, width, layers):
#     """Calculate scale factors for all auxiliary layers"""
#     return {
#         'eca': (height / layers[0].shape[0], width / layers[0].shape[1]),
#         'bare': (height / layers[1].shape[0], width / layers[1].shape[1]),
#         'ghsl': (height / layers[2].shape[0], width / layers[2].shape[1]),
#         'swf': (height / layers[3].shape[0], width / layers[3].shape[1]),
#         'rsd': (height / layers[4].shape[0], width / layers[4].shape[1]),
#         'nt': (height / layers[5].shape[0], width / layers[5].shape[1]),
#         'sv': (height / layers[6].shape[0], width / layers[6].shape[1])
#     }

# def process_chunk(lc, row_start, layers, scales, thresholds):
#     """Process a single chunk of land cover data"""
#     eca, bare, ghsl, swf, rsd, nt, sv = layers
#     chunk_intensity = lc.copy()
    
#     # Set No Data classes first
#     no_data_mask = np.isin(lc, NO_DATA_CLASSES)
#     chunk_intensity[no_data_mask] = NO_DATA_VALUE
    
#     # SV (Semi-Natural Vegetation)
#     sv_mask = np.isin(lc, [21, 22, 23, 24])
#     if np.any(sv_mask):
#         rows, cols = np.where(sv_mask)
#         global_rows = rows + row_start
#         eca_rows = np.clip((global_rows / scales['eca'][0]).astype(int), 0, eca.shape[0] - 1)
#         eca_cols = np.clip((cols / scales['eca'][1]).astype(int), 0, eca.shape[1] - 1)
#         eca_vals = eca[eca_rows, eca_cols]
#         t = thresholds['sv']
#         intensity = np.where(eca_vals > t['high'], 0, np.where(eca_vals >= t['low'], 1, 2))
#         chunk_intensity[rows, cols] = lc[rows, cols] * 10 + intensity
    
#     # Plantation
#     pl_mask = (lc == 3)
#     if np.any(pl_mask):
#         rows, cols = np.where(pl_mask)
#         global_rows = rows + row_start
#         sv_rows = np.clip((global_rows / scales['sv'][0]).astype(int), 0, sv.shape[0] - 1)
#         sv_cols = np.clip((cols / scales['sv'][1]).astype(int), 0, sv.shape[1] - 1)
#         sv_vals = sv[sv_rows, sv_cols]
#         t = thresholds['plantation']
#         intensity = np.where(sv_vals < t['low'], 2, np.where(sv_vals <= t['high'], 1, 0))
#         chunk_intensity[rows, cols] = 30 + intensity
    
#     # Crop
#     crop_mask = np.isin(lc, [6, 7])
#     if np.any(crop_mask):
#         rows, cols = np.where(crop_mask)
#         global_rows = rows + row_start
#         bare_rows = np.clip((global_rows / scales['bare'][0]).astype(int), 0, bare.shape[0] - 1)
#         bare_cols = np.clip((cols / scales['bare'][1]).astype(int), 0, bare.shape[1] - 1)
#         bare_vals = bare[bare_rows, bare_cols]
#         swf_rows = np.clip((global_rows / scales['swf'][0]).astype(int), 0, swf.shape[0] - 1)
#         swf_cols = np.clip((cols / scales['swf'][1]).astype(int), 0, swf.shape[1] - 1)
#         swf_vals = swf[swf_rows, swf_cols]
#         t = thresholds['crop']
#         minimal = (bare_vals < t['bare_low']) & (swf_vals > t['swf_high'])
#         light = ((bare_vals < t['bare_low']) & (swf_vals <= t['swf_high'])) | \
#                 ((bare_vals < t['bare_high']) & (swf_vals > t['swf_high']))
#         intense = ((bare_vals >= t['bare_low']) & (swf_vals < t['swf_high'])) | (bare_vals >= t['bare_high'])
#         intensity = np.where(minimal, 0, np.where(light, 1, 2))
#         chunk_intensity[rows, cols] = 70 + intensity
    
#     # Pasture
#     pasture_mask = (lc == 41)
#     if np.any(pasture_mask):
#         rows, cols = np.where(pasture_mask)
#         global_rows = rows + row_start
#         rsd_rows = np.clip((global_rows / scales['rsd'][0]).astype(int), 0, rsd.shape[0] - 1)
#         rsd_cols = np.clip((cols / scales['rsd'][1]).astype(int), 0, rsd.shape[1] - 1)
#         rsd_vals = rsd[rsd_rows, rsd_cols]
#         t = thresholds['pasture']
#         intensity = np.where(rsd_vals < t['low'], 0, np.where(rsd_vals <= t['high'], 1, 2))
#         chunk_intensity[rows, cols] = 60 + intensity
    
#     # Urban
#     urban_mask = np.isin(lc, [1, 9])
#     if np.any(urban_mask):
#         rows, cols = np.where(urban_mask)
#         global_rows = rows + row_start
#         nt_rows = np.clip((global_rows / scales['nt'][0]).astype(int), 0, nt.shape[0] - 1)
#         nt_cols = np.clip((cols / scales['nt'][1]).astype(int), 0, nt.shape[1] - 1)
#         nt_vals = nt[nt_rows, nt_cols]
#         ghsl_rows = np.clip((global_rows / scales['ghsl'][0]).astype(int), 0, ghsl.shape[0] - 1)
#         ghsl_cols = np.clip((cols / scales['ghsl'][1]).astype(int), 0, ghsl.shape[1] - 1)
#         ghsl_vals = ghsl[ghsl_rows, ghsl_cols]
#         t = thresholds['urban']
#         nt_class = np.where(nt_vals < t['nt_thresh1'], 4,
#                            np.where(nt_vals < t['nt_thresh2'], 3,
#                                    np.where(nt_vals < t['nt_thresh3'], 2, 1)))
#         intensity = np.where((nt_class > 2) & (ghsl_vals < t['ghsl_thresh1']), 0,
#                     np.where((nt_class > 2) & (ghsl_vals >= t['ghsl_thresh1']), 1,
#                     np.where((nt_class == 2) & (ghsl_vals < t['ghsl_thresh2']), 1,
#                     np.where((nt_class == 2) & (ghsl_vals >= t['ghsl_thresh2']), 2,
#                     np.where(nt_class == 1, 2, 0)))))
#         chunk_intensity[rows, cols] = 10 + intensity
    
#     # Urban minimal
#     if np.any(lc == 31):
#         chunk_intensity[lc == 31] = 10
    
#     return chunk_intensity

# # ============================================
# # Main Processing Loop
# # ============================================
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# for country in countries:
#     for year in years:
#         print(f"\n{'='*60}")
#         print(f"Processing {country.upper()} {year}")
#         print('='*60)
        
#         # Get file paths from already-loaded FILES dictionary
#         file_paths = FILES[country][year]
        
#         # Load auxiliary layers
#         print("Loading auxiliary layers...")
#         layers = load_auxiliary_layers(file_paths)
        
#         # Process
#         print("Starting classification...")
#         with rasterio.open(file_paths['lc']) as lc_src:
#             profile = lc_src.profile.copy()
#             profile.update(compress='lzw', nodata=NO_DATA_VALUE)
#             height, width = lc_src.shape
            
#             scales = calculate_scale_factors(height, width, layers)
#             print(f"Scale factors: ECA={scales['eca'][0]:.1f}x, RSD={scales['rsd'][0]:.1f}x, NT={scales['nt'][0]:.1f}x, SV={scales['sv'][0]:.1f}x")
            
#             output_file = OUTPUT_DIR / f"lu_intensity_{country}_{year}_{version}.tif"
            
#             with rasterio.open(output_file, 'w', **profile) as dst:
#                 for row_start in range(0, height, CHUNK_SIZE):
#                     row_end = min(row_start + CHUNK_SIZE, height)
#                     print(f"  Rows {row_start}-{row_end}/{height}")
                    
#                     window = Window(0, row_start, width, row_end - row_start)
#                     lc = lc_src.read(1, window=window)
                    
#                     chunk_intensity = process_chunk(lc, row_start, layers, scales, THRESHOLDS)
#                     dst.write(chunk_intensity, 1, window=window)
                    
#                     del lc, chunk_intensity
#                     gc.collect()
        
#         print(f"✅ Saved: {output_file}")
#         del layers
#         gc.collect()

# print("\n🎯 All processing complete!")


Processing DNK 2018
Loading auxiliary layers...
Starting classification...
Scale factors: ECA=100.2x, RSD=30.0x, NT=31.5x, SV=10.0x
  Rows 0-2000/35361
  Rows 2000-4000/35361
  Rows 4000-6000/35361
  Rows 6000-8000/35361
  Rows 8000-10000/35361
  Rows 10000-12000/35361
  Rows 12000-14000/35361
  Rows 14000-16000/35361
  Rows 16000-18000/35361
  Rows 18000-20000/35361
  Rows 20000-22000/35361
  Rows 22000-24000/35361
  Rows 24000-26000/35361
  Rows 26000-28000/35361
  Rows 28000-30000/35361
  Rows 30000-32000/35361
  Rows 32000-34000/35361
  Rows 34000-35361/35361
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/v2/lu_intensity_dnk_2018_v2.tif

Processing DNK 2019
Loading auxiliary layers...
Starting classification...
Scale factors: ECA=100.2x, RSD=30.0x, NT=31.5x, SV=10.0x
  Rows 0-2000/35361
  Rows 2000-4000/35361
  Rows 4000-6000/35361
  Rows 6000-8000/35361
  Rows 8000-10000/35361
  Rows 10000-12000/35361
  Rows 12000-14000/35361
  Rows 14000-16000/35361
  Rows 16000

## Create 1km Grid cell with proportions

### Multiple LU files

In [6]:
##___Configuration___##
version = 'v3'
countries = ['dnk', 'nld']
years = [2018, 2019, 2020, 2021, 2022, 2023]

BLOCK = 100   # 10 pixels = 100m or 100 for 1km grid

# Directories
input_dir = base_path / "BII_LU_layer" / "Land_use_map" / version
output_dir = base_path / "BII_LU_layer" / "Land_use_proportions" / version
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================
# Define intensity classes
# ============================================
INTENSITY_CLASSES = {
    'urban_minimal': 10,
    'urban_light': 11,
    'urban_intense': 12,
    'plantation_minimal': 30,
    'plantation_light': 31,
    'plantation_intense': 32,
    'pasture_minimal': 60,
    'pasture_light': 61,
    'pasture_intense': 62,
    'crop_minimal': 70,
    'crop_light': 71,
    'crop_intense': 72,
    'sv_mature_minimal': 210,
    'sv_mature_light': 211,
    'sv_mature_intense': 212,
    'sv_intermediate_minimal': 220,
    'sv_intermediate_light': 221,
    'sv_intermediate_intense': 222,
    'sv_young_minimal': 230,
    'sv_young_light': 231,
    'sv_young_intense': 232,
    'sv_indeterminate_minimal': 240,
    'sv_indeterminate_light': 241,
    'sv_indeterminate_intense': 242,
}


# ============================================
# Process all countries and years
# ============================================
for country in countries:
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing {country.upper()} {year}")
        print('='*60)
        
        # Load intensity file
        input_file = input_dir / f"lu_intensity_{country}_{year}_{version}.tif"
        
        if not input_file.exists():
            print(f"⚠️  File not found: {input_file}")
            continue
        
        print(f"Loading: {input_file.name}")
        
        with rasterio.open(input_file) as src:
            intensity = src.read(1)
            height, width = src.shape
            
            # 1km grid dimensions (100 pixels = 1km at 10m resolution)
            grid_height = height // BLOCK
            grid_width = width // BLOCK
            
            print(f"Original: {height} x {width} (10m)")
            print(f"Grid: {grid_height} x {grid_width} (1km)")
            
            # Store profile for output
            transform = src.transform * src.transform.scale(BLOCK, BLOCK)
            output_profile = src.profile.copy()
            output_profile.update({
                'height': grid_height,
                'width': grid_width,
                'transform': transform,
                'dtype': 'float32',
                'count': len(INTENSITY_CLASSES),
                'compress': 'lzw',
                'nodata': -9999
            })
            
            # Calculate proportions for all classes
            print("Calculating proportions...")
            proportion_grids = {}
            
            for class_name, class_value in INTENSITY_CLASSES.items():
                print(f"  {class_name}...", end='')
                
                grid = np.zeros((grid_height, grid_width), dtype=np.float32)
                
                # Aggregate to 1km
                for i in range(grid_height):
                    for j in range(grid_width):
                        row_start = i * BLOCK
                        col_start = j * BLOCK
                        row_end = min(row_start + BLOCK, height)
                        col_end = min(col_start + BLOCK, width)
                        
                        block = intensity[row_start:row_end, col_start:col_end]
                        
                        # Exclude no data (255) from calculations
                        valid_pixels = block[block != 255]
                        
                        if valid_pixels.size > 0:
                            class_pixels = np.sum(valid_pixels == class_value)
                            grid[i, j] = class_pixels / valid_pixels.size
                        else:
                            grid[i, j] = -9999  # No valid data
                
                proportion_grids[class_name] = grid
                
                # Quick stats
                valid_cells = np.sum(grid >= 0)
                non_zero = np.sum(grid > 0)
                print(f" {non_zero}/{valid_cells} cells")
        
        # Save as multi-band GeoTIFF
        output_file = output_dir / f"lu_proportions_{country}_{year}_{version}_{BLOCK}0m.tif"
        
        print(f"\nSaving: {output_file.name}")
        with rasterio.open(output_file, 'w', **output_profile) as dst:
            for i, (class_name, grid) in enumerate(proportion_grids.items(), 1):
                dst.write(grid, i)
                dst.set_band_description(i, class_name)
        
        print(f"✅ Saved with {len(INTENSITY_CLASSES)} bands")

print("\n🎯 All proportions calculated and saved!")


Processing DNK 2018
Loading: lu_intensity_dnk_2018_v3.tif
Original: 35361 x 45093 (10m)
Grid: 353 x 450 (1km)
Calculating proportions...
  urban_minimal... 44671/158805 cells
  urban_light... 3410/158805 cells
  urban_intense... 549/158805 cells
  plantation_minimal... 44169/158805 cells
  plantation_light... 32467/158805 cells
  plantation_intense... 20670/158805 cells
  pasture_minimal... 38813/158805 cells
  pasture_light... 19643/158805 cells
  pasture_intense... 15423/158805 cells
  crop_minimal... 34785/158805 cells
  crop_light... 45773/158805 cells
  crop_intense... 40317/158805 cells
  sv_mature_minimal... 1832/158805 cells
  sv_mature_light... 12139/158805 cells
  sv_mature_intense... 34158/158805 cells
  sv_intermediate_minimal... 1912/158805 cells
  sv_intermediate_light... 12948/158805 cells
  sv_intermediate_intense... 36920/158805 cells
  sv_young_minimal... 1851/158805 cells
  sv_young_light... 12658/158805 cells
  sv_young_intense... 35587/158805 cells
  sv_indetermin